# **Prévalence du paludisme chez les enfants**
# Données issues des Enquêtes Démographiques et de Santé (EDS - DHS)

Ce rapport génère des visualisations pour l'indicateur relatif au **pourcentage d'enfants âgés de 6 à 59 mois, testés positif au paludisme** via un test de diagnostic rapide (TDR) ou via microscopie, à partir des données de l'Enquête Démographique et de Santé (EDS - DHS).

---

* *Numérateur* : Nombre d'enfants qui sont membres de fait du ménage, testés par test de diagnostic rapide (TDR) ou par microscopie, et dont le résultat est positif pour le paludisme.
* *Dénominateur* : Nombre d'enfants qui sont membres de fait du ménage, testés par test de diagnostic rapide (TDR) ou par microscopie.

---

Pour plus d'informations (en anglais):
- Ressources relatives à la prévalence du paludisme chez les enfants
    - [Définition et calculs](https://dhsprogram.com/data/Guide-to-DHS-Statistics/index.htm#t=Prevalence_of_Malaria_in_Children.htm%23Percentage_of_children22bc-1&rhtocid=_15_13_0)
- [Les questionnaires utilisés dans les EDS/DHS](https://dhsprogram.com/publications/publication-dhsg4-dhs-questionnaires-and-manuals.cfm)

---

*Note* : Contrairement à la majorité des analyses dans le cadre du processus SNT, cette analyse est menée au niveau administratif **ADM1**, en raison de la disponibilité des données.

## 1. Configuration

In [ ]:
rm(list = ls())

options(scipen=999)

# Global paths
Sys.setenv(PROJ_LIB = "/opt/conda/share/proj")
Sys.setenv(GDAL_DATA = "/opt/conda/share/gdal")

In [ ]:
# Paths
ROOT_PATH <- '~/workspace'
PIPELINE_PATH <- file.path(ROOT_PATH, 'pipelines', 'snt_dhs_indicators')
CONFIG_PATH <- file.path(ROOT_PATH, 'configuration')
CODE_PATH <- file.path(ROOT_PATH, 'code')
DATA_PATH <- file.path(ROOT_PATH, 'data')
DHS_DATA_PATH <- file.path(DATA_PATH, 'dhs', 'raw')
OUTPUT_DATA_PATH <- file.path(DATA_PATH, 'dhs', 'indicators', 'prevalence')
OUTPUT_PLOTS_PATH <- file.path(ROOT_PATH, 'pipelines', 'snt_dhs_indicators', 'reporting', 'outputs')

In [ ]:
# Load notebook-specific utilities
source(file.path(CODE_PATH, "snt_utils.r"))
source(file.path(CODE_PATH, "snt_report.r"))
source(file.path(CODE_PATH, "snt_palettes.r"))
source(file.path(PIPELINE_PATH, "utils", "snt_dhs_prevalence_report.r"))

# List required pcks
required_packages <- c("sf", "glue", "data.table", "ggplot2", "stringi", "jsonlite", "httr", "reticulate", "arrow", "IRdisplay")

# Execute function
install_and_load(required_packages)

Sys.setenv(RETICULATE_PYTHON = "/opt/conda/bin/python")
reticulate::py_config()$python
openhexa <- import("openhexa.sdk")

# Load SNT config
CONFIG_FILE_NAME <- "SNT_config.json"
config_json <- tryCatch({ fromJSON(file.path(CONFIG_PATH, CONFIG_FILE_NAME)) },
                        error = function(e) {
                          msg <- paste0("Error while loading configuration", conditionMessage(e))
                          cat(msg)
                          stop(msg)
                        })

msg <- paste0("SNT configuration loaded from : ", file.path(CONFIG_PATH, CONFIG_FILE_NAME))
log_msg(msg)

# Set config variables
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE


In [ ]:
data_source <- 'DHS'
admin_level <- 'ADM1'
admin_id_col <- glue(admin_level, 'ID', .sep='_')
admin_name_col <- glue(admin_level, 'NAME', .sep='_')
admin_cols <- c(admin_id_col, admin_name_col)

## 2. Chargement et pré-processing des données à visualiser

**Les données utilisées**

Toutes les données utilisées dans ce rapport sont agrégées au niveau administratif ADM1:

* Données spatiales : fond de carte (DHIS2)
* Données calculées par le pipeline :
    - valeurs estimées de prévalence du paludisme chez les enfants de moins de 5 ans, ainsi que sur la précision statistique de cette estimation (intervalles de confiance à 95%)

In [ ]:
# Load spatial file from dataset

dhis2_dataset <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED

spatial_data_filename <- paste(COUNTRY_CODE, "shapes.geojson", sep = "_")
# spatial_data <- read_sf(file.path(DATA_PATH, 'dhis2', 'formatted', spatial_data_filename))
spatial_data <- get_latest_dataset_file_in_memory(dhis2_dataset, spatial_data_filename)
log_msg(glue("File {spatial_data_filename} successfully loaded from dataset version: {dhis2_dataset}"))

spatial_data <- st_as_sf(spatial_data)

# aggregate geometries by the admin columns
spatial_data <- aggregate_geometry(
  sf_data=spatial_data,
  admin_id_colname=admin_id_col,
  admin_name_colname=admin_name_col
)

# keep class
spatial_data <- st_as_sf(spatial_data)

if(COUNTRY_CODE == "COD"){
  spatial_data[[admin_name_col]] <- clean_admin_names(spatial_data[[admin_name_col]])
}

## 3. Création de graphiques

Le rapport crée deux visualisations:
   - Une carte choroplèthe indiquant l'estimation moyenne de la valeur de l'indicateur, sur base des données d'enquête
   - Un graphique de son intervalle de confiance, avec des barres d’erreur pour chaque ADM1

In [ ]:

indicator_u5prev <- 'PCT_U5_PREV_RDT'

In [ ]:
filename_without_extension <- glue("{COUNTRY_CODE}_{data_source}_{admin_level}_{toupper(indicator_u5prev)}")
u5_rdt_prevalence_table <- fread(file.path(OUTPUT_DATA_PATH, paste0(filename_without_extension, '.csv')))

plot_data =  merge(spatial_data, u5_rdt_prevalence_table, by = admin_cols, all = TRUE)

In [ ]:
lower_bound_col <- glue("{toupper(indicator_u5prev)}_CI_LOWER_BOUND")
upper_bound_col <- glue("{toupper(indicator_u5prev)}_CI_UPPER_BOUND")
sample_avg_col <- glue("{toupper(indicator_u5prev)}_SAMPLE_AVERAGE")

In [ ]:
prev_plot <- make_pct_choropleth_map(
  map_data = plot_data,
  target_colname = sample_avg_col,
  plot_title = "Prévalence du paludisme chez les enfants (%)",
  plot_subtitle = COUNTRY_CODE,
  plot_caption = glue("Données: {data_source}")
)

In [ ]:
prev_plot_filename <- glue("{COUNTRY_CODE}_{data_source}_{admin_level}_{toupper(indicator_u5prev)}_plot.png")
prev_plot_path <- file.path(OUTPUT_PLOTS_PATH, prev_plot_filename)
suppressMessages(ggsave(prev_plot, file = prev_plot_path, width = 6, height = 5, dpi = 300))

In [ ]:
display_png(file = prev_plot_path)

In [ ]:
prev_ci_plot_title <- glue("{COUNTRY_CODE} {data_source} {toupper(indicator_u5prev)} CI")
prev_ci_plot_xlab <- admin_level
prev_ci_plot_ylab <- glue("Enfants testés positif au paludisme (%)")

In [ ]:
prev_ci_plot <- make_ci_plot(
  df_to_plot=plot_data,
  admin_colname=admin_name_col,
  point_estimation_colname=sample_avg_col,
  ci_lower_colname=lower_bound_col,
  ci_upper_colname=upper_bound_col,
  plot_title=prev_ci_plot_title,
  plot_subtitle=COUNTRY_CODE,
  plot_caption=glue("Données: {data_source}"),
  x_title=prev_ci_plot_xlab,
  y_title=prev_ci_plot_ylab
)


In [ ]:
prev_ci_plot_filename <- glue("{COUNTRY_CODE}_{data_source}_{admin_level}_{toupper(indicator_u5prev)}_prev_ci_plot.png")
prev_ci_plot_path <- file.path(OUTPUT_PLOTS_PATH, prev_ci_plot_filename)
suppressMessages(ggsave(filename=prev_ci_plot_path, plot=prev_ci_plot, width = 6, height = 5, dpi = 300))

In [ ]:
display_png(file = prev_ci_plot_path)